# 花店情境課件 02：訂單流程與例外處理
Status: in_use  
Prereq: list/dict、函式、模組化基本、能獨立跑小專題  
Can-do checklist:  
- [ ] 能把需求拆成資料結構與操作函式
- [ ] 能用小測資驗證每個操作
- [ ] 能把程式整理成可重跑的主程式/模組


定位：延續第一節的資料模型，建立「訂單 → 扣庫存 → 計算金額」的完整流程。

## 學習目標
- 建立訂單資料結構
- 實作下單流程（含庫存檢查）
- 加入基礎例外處理，避免程式崩潰

## 1. 基礎庫存資料

In [1]:
# inventory: dict of dict
inventory = {
    "玫瑰": {"cost": 15, "price": 30, "stock": 20},
    "百合": {"cost": 18, "price": 36, "stock": 12},
    "康乃馨": {"cost": 10, "price": 20, "stock": 25},
}

## 2. 訂單資料結構
- 訂單用 dict 表示
- items 用 list of tuple (花名, 數量)

In [2]:
order = {
    "id": "OD001",
    "items": [("玫瑰", 2), ("百合", 1)],
    "total": 0,
    "status": "created",
}

order

{'id': 'OD001',
 'items': [('玫瑰', 2), ('百合', 1)],
 'total': 0,
 'status': 'created'}

## 3. 檢查庫存與扣庫存

In [3]:
def check_stock(inv, items):
    for name, qty in items:
        if name not in inv:
            return False, f"查無品項：{name}"
        if inv[name]["stock"] < qty:
            return False, f"庫存不足：{name}"
    return True, "ok"


def apply_order(inv, items):
    for name, qty in items:
        inv[name]["stock"] -= qty

## 4. 計算訂單金額與毛利

In [4]:
def calc_bill(inv, items):
    total = 0
    profit = 0
    for name, qty in items:
        total += inv[name]["price"] * qty
        profit += (inv[name]["price"] - inv[name]["cost"]) * qty
    return total, profit

## 5. 完整下單流程

In [5]:
def place_order(inv, order):
    ok, msg = check_stock(inv, order["items"])
    if not ok:
        order["status"] = "rejected"
        return False, msg

    apply_order(inv, order["items"])
    total, profit = calc_bill(inv, order["items"])
    order["total"] = total
    order["status"] = "paid"
    return True, (total, profit)

print(place_order(inventory, order))
order

(True, (96, 48))


{'id': 'OD001', 'items': [('玫瑰', 2), ('百合', 1)], 'total': 96, 'status': 'paid'}

## 6. 例外處理（避免程式崩潰）

In [6]:
try:
    bad_order = {"id": "OD002", "items": [("蘭花", 1)], "total": 0, "status": "created"}
    print(place_order(inventory, bad_order))
except Exception as e:
    print("發生錯誤：", e)

(False, '查無品項：蘭花')


## 6-1. 導入 NamedTuple / dataclass（進階版本）
當訂單與庫存開始變複雜，改用具名欄位會更清楚。


In [7]:
from typing import NamedTuple
from dataclasses import dataclass

class Item(NamedTuple):
    name: str
    cost: int
    price: int

@dataclass
class Stock:
    name: str
    qty: int

# 固定資料用 NamedTuple，狀態用 dataclass
rose = Item("玫瑰", 15, 30)
rose_stock = Stock("玫瑰", 20)
rose, rose_stock


(Item(name='玫瑰', cost=15, price=30), Stock(name='玫瑰', qty=20))

## 7. 小練習
1. 加入折扣機制：訂單總價超過 500 打 9 折。
2. 設計「部分可出貨」：有缺貨時，回傳可以出貨的清單。
3. 新增訂單列表 orders，把每筆訂單加入後再統計今日營收。

## 8. 小結
- 訂單流程的關鍵是：檢查 → 扣庫存 → 計算 → 更新狀態
- 例外處理可以保護程式不崩潰